# Animal Classification — 5 Species (CNN Multi-Class)
Runtime > Change runtime type > T4 GPU recommended.

In [ ]:
!pip install -q kaggle tensorflow scikit-learn matplotlib

## 1. Kaggle token se dataset download karo

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = input("Apna Kaggle API token yahan paste karo aur Enter dabao: ")
!kaggle datasets download -d miadul/animal-image-classification-5-species -p data_raw --unzip

## 2. Dekho dataset kaise organized hai

In [ ]:
!find data_raw -maxdepth 3 -type d

## 3. train/test split banao
Ye dataset usually already class-wise folders mein hota hai (e.g. `data_raw/Cat`, `data_raw/Dog`, ...).
Neeche wala code un class-folders ko dhoondh kar khud 80/20 train/test split kar dega — koi manual sorting nahi karni.

In [ ]:
import os, glob, shutil, random
random.seed(42)

# Step 2 ke output se class folders ka asli parent path pata chalega — zaroorat ho to yahan edit karo
SOURCE_ROOT = 'data_raw'  # <-- agar images ek extra subfolder ke andar hain (e.g. data_raw/animals), yahan update karo

class_folders = [d for d in glob.glob(f'{SOURCE_ROOT}/*') if os.path.isdir(d)]
print('Classes found:', [os.path.basename(c) for c in class_folders])

for cls_path in class_folders:
    cls_name = os.path.basename(cls_path)
    images = glob.glob(f'{cls_path}/*.*')
    random.shuffle(images)
    split = int(0.8 * len(images))

    os.makedirs(f'data/train/{cls_name}', exist_ok=True)
    os.makedirs(f'data/test/{cls_name}', exist_ok=True)

    for p in images[:split]:
        shutil.copy(p, f'data/train/{cls_name}/')
    for p in images[split:]:
        shutil.copy(p, f'data/test/{cls_name}/')

    print(f'{cls_name}: {len(images)} images ({split} train / {len(images)-split} test)')

## 4. tf.data pipelines banao

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    'data/train', validation_split=0.2, subset='training', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')

val_ds = tf.keras.utils.image_dataset_from_directory(
    'data/train', validation_split=0.2, subset='validation', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')

test_ds = tf.keras.utils.image_dataset_from_directory(
    'data/test', image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=False)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print('Classes:', class_names)

norm = tf.keras.layers.Rescaling(1./255)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x,y: (norm(x), y)).cache().prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda x,y: (norm(x), y)).cache().prefetch(AUTOTUNE)
test_ds = test_ds.map(lambda x,y: (norm(x), y)).prefetch(AUTOTUNE)

## 5. CNN model banao (multi-class: softmax output)

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),
    augmentation,
    tf.keras.layers.Conv2D(32,3,activation='relu',padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64,3,activation='relu',padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128,3,activation='relu',padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(256,3,activation='relu',padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),  # multi-class output
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 6. Train karo

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=25, callbacks=callbacks)

## 7. Evaluate karo

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(history.history['accuracy'], label='train_acc')
axes[0].plot(history.history['val_accuracy'], label='val_acc')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train_loss')
axes[1].plot(history.history['val_loss'], label='val_loss')
axes[1].set_title('Loss'); axes[1].legend()
plt.savefig('training_curves.png')
plt.show()

results = model.evaluate(test_ds, return_dict=True)
print(results)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

## 8. Model save karo aur predict karo

In [ ]:
model.save('final_model.keras')

def predict_image(path):
    img = tf.keras.utils.load_img(path, target_size=(224,224))
    arr = tf.keras.utils.img_to_array(img)/255.0
    arr = np.expand_dims(arr, 0)
    preds = model.predict(arr, verbose=0)[0]
    idx = np.argmax(preds)
    print(f'{class_names[idx]} ({preds[idx]:.2%} confidence)')

# predict_image('path/to/your/test_image.jpg')